# 03 — Evaluation: does any of this actually work?

Everything before this notebook was mechanism. This one asks whether the mechanism
*helps*, and answers it with numbers instead of intuition.

Corpus: **ESCI** (Amazon Shopping Queries) — 482,060 products, 8,956 test queries,
181,701 graded judgments. Prerequisites:

```bash
.venv/bin/python -m opensearch_demo.esci                                    # ~5 min
.venv/bin/python -m opensearch_demo.demo --dataset esci --rebuild "warm up" # ~90 min
```

In [ ]:
import random, time
import pandas as pd
from opensearch_demo import get_client, esci
from opensearch_demo.search import lexical_search, neural_search, hybrid_search
from opensearch_demo.rerank import rerank
from opensearch_demo.evaluate import evaluate_run, evaluate_query, ndcg_at_k

client = get_client()
qrels = esci.load_qrels()
print(f"{len(qrels):,} test queries with judgments")
print(f"{client.count(index=esci.INDEX_NAME)['count']:,} products indexed")

## What we are measuring

**NDCG@k** — did we put the good documents near the top?

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{2^{rel_i} - 1}{\log_2(i+1)}
\qquad \text{NDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}$$

The exponential gain means the gap between *exact match* and *substitute* counts for
far more than the gap between *substitute* and *complement*. Dividing by the ideal
ranking's DCG normalises to [0, 1], so queries with many relevant products are
comparable to queries with few.

**recall@k** — did retrieval find the good documents *at all*, anywhere in the top k?

That distinction is the spine of this notebook: **NDCG measures ordering, recall
measures presence.** Reranking can only ever improve the first.

### One query, up close

Before averaging over hundreds of queries, look at what a single judgment set is.

In [ ]:
qid = sorted(qrels)[7]
q = qrels[qid]
print(f"query: {q['query']!r}   ({len(q['judgments'])} judged products)\n")
labels = pd.Series([j["label"] for j in q["judgments"].values()]).value_counts()
print(labels.to_string())
print("\nE=Exact  S=Substitute  C=Complement  I=Irrelevant")

In [ ]:
# What our pipeline returns for it, scored against those judgments
hits = hybrid_search(client, q["query"], k=10, index=esci.INDEX_NAME)
rows = []
for rank, h in enumerate(hits, 1):
    j = q["judgments"].get(h["id"])
    rows.append({"rank": rank, "label": j["label"] if j else "—unjudged—",
                 "title": h["title"][:62]})
display(pd.DataFrame(rows).set_index("rank"))
print(evaluate_query([h["id"] for h in hits], q["judgments"]))

Note the `—unjudged—` rows. Nobody rated those products for this query, and we
count them as non-relevant. That is the standard closed-world assumption, and it means
**every number here is a floor**: fine for comparing our own variants against each
other, not comparable to a published leaderboard over a different corpus.

## The comparison

Four variants, same queries, same judgments. This is the experiment.

Reduce `N_QUERIES` if you want it to finish faster — 150 takes a couple of minutes,
most of it the cross-encoder.

In [ ]:
N_QUERIES, RETRIEVE_K, RERANK_DEPTH = 150, 100, 25

qids = sorted(qrels); random.Random(11).shuffle(qids)
sample = qids[:N_QUERIES]

def collect(fn, rerank_it=False):
    run, t0 = {}, time.perf_counter()
    for qid in sample:
        text = qrels[qid]["query"]
        hits = fn(client, text, k=RETRIEVE_K, index=esci.INDEX_NAME)
        if rerank_it:
            hits = rerank(text, hits[:RERANK_DEPTH]) + hits[RERANK_DEPTH:]
        run[qid] = [h["id"] for h in hits]
    return run, (time.perf_counter() - t0) / len(sample)

variants = [
    ("lexical (BM25)",  lexical_search, False),
    ("neural (k-NN)",   neural_search,  False),
    ("hybrid (RRF)",    hybrid_search,  False),
    ("hybrid + rerank", hybrid_search,  True),
]
rows = []
for name, fn, rr in variants:
    run, secs = collect(fn, rr)
    m = evaluate_run(run, qrels)
    rows.append({"variant": name, **{k: round(v, 4) for k, v in m.items()},
                 "sec/query": round(secs, 3)})
df = pd.DataFrame(rows).set_index("variant")
df

## Reading the table

**1 · BM25 beats the vectors.** Plain keyword search outscores neural retrieval by a
clear margin. Three reasons, all real:

- product search is unusually lexical — brands, model numbers, exact product nouns
- our embedding model is a general-purpose MiniLM that has never seen a product catalogue
- 42% of this corpus exceeds the 256-token window and got truncated at index time

This is worth sitting with, because the fashionable assumption is the opposite. Vectors
are not automatically better; they are better *at particular things*.

**2 · Hybrid beats both — including the retriever that loses on its own.** That is the
entire argument for RRF. The two retrievers fail on *different* queries, so fusing them
recovers documents neither would have ranked highly alone. A weak retriever still
contributes if its mistakes are uncorrelated with the other's.

**3 · Reranking is the single biggest win**, and it is the most expensive step. Note the
exchange rate in `sec/query`.

In [ ]:
base = df.loc["lexical (BM25)"]
comp = pd.DataFrame({
    "NDCG@10 vs BM25": ((df["ndcg@10"] / base["ndcg@10"] - 1) * 100).round(1),
    "latency vs BM25": (df["sec/query"] / base["sec/query"]).round(1).astype(str) + "x",
})
comp

## The most instructive number in the table

Compare `hybrid` and `hybrid + rerank` on **recall@100**. They are *identical*.

Reranking changed NDCG@10 substantially while recall@100 did not move at all — because
reranking **reorders what retrieval already found and never retrieves anything new**.
It cannot rescue a document that retrieval missed.

Which makes the recall ceiling the thing to worry about:

In [ ]:
ceiling = df["recall@100"].max()
print(f"best recall@100 = {ceiling:.4f}")
print(f"-> {(1-ceiling)*100:.0f}% of relevant products never enter the top 100 at all.")
print("   No reranker, however good, can recover them.")
print("   Fixing that means better retrieval: a domain-tuned embedding model,")
print("   a longer context window, or learned sparse retrieval.")

## Where the improvements would come from

Everything above uses off-the-shelf models. The obvious next moves:

| Change | Attacks | Session |
|---|---|---|
| Fine-tune the embedding model on ESCI train | the recall ceiling | loss functions (next) |
| Longer-window model (42% truncation) | recall + ranking | — |
| Tune RRF weights against these judgments | ranking | now, see below |
| Deeper reranking | ranking, costs latency | now, see below |

The train split is already on disk — `esci.load_examples()` with `split == "train"` gives
20,888 queries and 419,653 judgments to fine-tune on. After that, every vector must be
recomputed and the index rebuilt, because query and document embeddings must come from
the same model.

### Exercise 1 — tune the fusion weights

We fused BM25 and k-NN evenly. Since BM25 is the stronger retriever here, weighting it
higher should help. Find out.

In [ ]:
from opensearch_demo.search import reciprocal_rank_fusion

def weighted_run(w_lex, w_vec, n=60):
    run = {}
    for qid in sample[:n]:
        text = qrels[qid]["query"]
        lex = lexical_search(client, text, k=RETRIEVE_K, index=esci.INDEX_NAME)
        vec = neural_search(client, text, k=RETRIEVE_K, index=esci.INDEX_NAME)
        run[qid] = [h["id"] for h in
                    reciprocal_rank_fusion([lex, vec], k=60, weights=[w_lex, w_vec])]
    return evaluate_run(run, qrels)["ndcg@10"]

sweep = pd.DataFrame([{"lex:vec": f"{a}:{b}", "ndcg@10": round(weighted_run(a, b), 4)}
                      for a, b in [(1,0), (2,1), (1,1), (1,2), (0,1)]]).set_index("lex:vec")
sweep

### Exercise 2 — how deep should reranking go?

Reranking depth trades latency for quality linearly. Find where your reranker stops
changing the top 10.

In [ ]:
rows = []
for depth in [5, 10, 25, 50]:
    run, secs = {}, time.perf_counter()
    for qid in sample[:60]:
        text = qrels[qid]["query"]
        hits = hybrid_search(client, text, k=RETRIEVE_K, index=esci.INDEX_NAME)
        run[qid] = [h["id"] for h in rerank(text, hits[:depth]) + hits[depth:]]
    m = evaluate_run(run, qrels)
    rows.append({"rerank depth": depth, "ndcg@10": round(m["ndcg@10"], 4),
                 "sec/query": round((time.perf_counter()-secs)/60, 3)})
pd.DataFrame(rows).set_index("rerank depth")

---
**Cross-check worth doing:** `ranx` (MIT) is the standard library for exactly these
metrics, and `ranx.fuse` implements RRF among ~20 fusion algorithms. Running our numbers
through it would validate both our metric implementation and our hand-rolled fusion.
`uv pip install ranx` — see `docs/research/tools.html`.